In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from datetime import datetime

from simulation.simulation import Simulation
from simulation.demand_engine import preview_day
import simulation.simulation_config as config

In [3]:
import pandas as pd

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation

In [4]:
import pandas as pd

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation

sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date="2023-01-01",
    end_date="2023-01-31",
)

In [5]:
po_df = pd.DataFrame(sim.purchase_orders)

if po_df.empty:
    print("No purchase orders were created during this period.")
else:
    print(po_df["po_status"].value_counts())
    display(po_df.head())

po_status
Open        49
Received     1
Name: count, dtype: int64


,purchase_order_id,product_id,supplier_id,supplier_name,order_date,expected_receipt_date,actual_receipt_date,lead_time_days,ordered_qty,received_qty,po_status
0,PO50000,1076,S001,Fox Factory,2023-01-10,2023-01-31,2023-01-31,21,65,65,Received
1,PO50001,1103,S001,Fox Factory,2023-01-11,2023-02-01,None,21,61,0,Open
2,PO50002,1108,S001,Fox Factory,2023-01-11,2023-02-01,None,21,60,0,Open
3,PO50003,1074,S001,Fox Factory,2023-01-12,2023-02-02,None,21,61,0,Open
4,PO50004,1099,S001,Fox Factory,2023-01-12,2023-02-02,None,21,60,0,Open


In [6]:
sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date="2023-01-01",
    end_date="2023-12-31",
)

orders_df = pd.DataFrame(sim.sales_orders)
lines_df = pd.DataFrame(sim.sales_order_lines)
po_df = pd.DataFrame(sim.purchase_orders)
daily_df = pd.DataFrame(sim.daily_order_summary)

print("Sales orders:", len(orders_df))
print("Sales order lines:", len(lines_df))
print("Purchase orders:", len(po_df))
print("Requested units:", lines_df["requested_qty"].sum())
print("Fulfilled units:", lines_df["fulfilled_qty"].sum())
print("Backordered units:", lines_df["backordered_qty"].sum())
print("Ending inventory:", sim.inventory["on_hand"].sum())

if not po_df.empty:
    print("Open PO units:", po_df.loc[po_df["po_status"] == "Open", "ordered_qty"].sum())
    print("Received PO units:", po_df.loc[po_df["po_status"] == "Received", "received_qty"].sum())

Sales orders: 9218
Sales order lines: 25567
Purchase orders: 770
Requested units: 51283
Fulfilled units: 45256
Backordered units: 6027
Ending inventory: 8637
Open PO units: 1846
Received PO units: 45493


In [7]:
initial_inventory_units = len(sim.products) * 40

received_units = 0

if not po_df.empty:
    received_units = po_df.loc[
        po_df["po_status"] == "Received",
        "received_qty"
    ].sum()

fulfilled_units = lines_df["fulfilled_qty"].sum()
ending_inventory_units = sim.inventory["on_hand"].sum()

inventory_difference = (
    initial_inventory_units
    + received_units
    - fulfilled_units
    - ending_inventory_units
)

inventory_difference

0

In [8]:
sim.export_tables()

In [9]:
from simulation.forecasting import (
    generate_baseline_forecast,
    calculate_forecast_metrics,
)

In [10]:
forecast_df = generate_baseline_forecast(
    sim,
    lookback_months=3,
)

forecast_df.head()

,forecast_month,product_id,model_name,category,lookback_months,forecast_qty,actual_qty,forecast_error,absolute_error,absolute_percentage_error,seasonality_adjustment
0,2023-04-01,1001,Catalina,Cross Country,3,9,5,-4,4,0.800000,1.714
1,2023-05-01,1001,Catalina,Cross Country,3,9,3,-6,6,2.000000,1.473
2,2023-06-01,1001,Catalina,Cross Country,3,6,7,1,1,0.142857,1.130
3,2023-07-01,1001,Catalina,Cross Country,3,4,5,1,1,0.200000,0.896
4,2023-08-01,1001,Catalina,Cross Country,3,4,7,3,3,0.428571,0.789


In [11]:
calculate_forecast_metrics(forecast_df)

{'forecast_rows': 1890,
 'total_actual_qty': 41820,
 'total_forecast_qty': 41374,
 'total_absolute_error': 11508,
 'wape': 0.2752,
 'bias_pct': 0.0107,
 'mean_absolute_percentage_error': 0.5174}

In [12]:
forecast_df.sort_values(
    "absolute_error",
    ascending=False,
).head(10)

,forecast_month,product_id,model_name,category,lookback_months,forecast_qty,actual_qty,forecast_error,absolute_error,absolute_percentage_error,seasonality_adjustment
955,2023-05-01,1107,Romero,Aggressive Trail,3,72,130,58,58,0.446154,1.473
991,2023-05-01,1111,Romero,Aggressive Trail,3,55,96,41,41,0.427083,1.473
1269,2023-04-01,1142,Oracle,Enduro,3,42,79,37,37,0.468354,1.714
731,2023-06-01,1082,Sabino,Trail,3,80,44,-36,36,0.818182,1.130
658,2023-05-01,1074,Sabino,Trail,3,80,116,36,36,0.310345,1.473
999,2023-04-01,1112,Romero,Aggressive Trail,3,79,44,-35,35,0.795455,1.714
1066,2023-08-01,1119,Romero,Aggressive Trail,3,33,67,34,34,0.507463,0.789
674,2023-12-01,1075,Sabino,Trail,3,40,74,34,34,0.459459,0.735
918,2023-04-01,1103,Romero,Aggressive Trail,3,56,90,34,34,0.377778,1.714
754,2023-11-01,1084,Sabino,Trail,3,44,78,34,34,0.435897,0.906


In [13]:
forecast_by_model = (
    forecast_df
    .groupby("model_name")
    .agg(
        actual_qty=("actual_qty", "sum"),
        forecast_qty=("forecast_qty", "sum"),
        absolute_error=("absolute_error", "sum"),
        forecast_error=("forecast_error", "sum"),
    )
    .reset_index()
)

forecast_by_model["wape"] = (
    forecast_by_model["absolute_error"]
    / forecast_by_model["actual_qty"]
)

forecast_by_model["bias_pct"] = (
    forecast_by_model["forecast_error"]
    / forecast_by_model["actual_qty"]
)

forecast_by_model.sort_values("wape", ascending=False)

,model_name,actual_qty,forecast_qty,absolute_error,forecast_error,wape,bias_pct
6,Sonoita,1541,1491,796,50,0.516548,0.032446
5,Sky Island,2543,2516,1059,27,0.416437,0.010617
2,Rincon,3468,3447,1387,21,0.399942,0.006055
0,Catalina,5945,5981,1602,-36,0.269470,-0.006056
1,Oracle,7143,7164,1845,-21,0.258295,-0.002940
3,Romero,10169,9969,2510,200,0.246829,0.019668
4,Sabino,11011,10806,2309,205,0.209699,0.018618


In [14]:
sim.export_tables()

In [15]:
import pandas as pd

from simulation.simulation import Simulation
from simulation.simulation_runner import run_sales_simulation
from simulation.forecasting import generate_baseline_forecast
from simulation.analytics import export_analytics_tables

In [16]:
sim = Simulation()
sim.load_master_data()
sim.initialize_inventory()

daily_summary = run_sales_simulation(
    sim,
    start_date="2021-01-01",
    end_date="2025-12-31",
)

forecast_df = generate_baseline_forecast(
    sim,
    lookback_months=3,
)

analytics_tables = export_analytics_tables(sim)

KeyboardInterrupt: 

In [ ]:
analytics_tables.keys()

In [ ]:
analytics_tables["monthly_sales_summary"].head()

In [ ]:
analytics_tables["model_performance_summary"].head()

In [ ]:
analytics_tables["inventory_kpi_summary"].head()

In [ ]:
analytics_tables["forecast_accuracy_by_model"].head()

In [ ]:
analytics_tables["supplier_performance_summary"].head()

In [ ]:
analytics_tables["daily_kpi_summary"].head()

In [ ]:
orders_df = pd.DataFrame(sim.sales_orders)
lines_df = pd.DataFrame(sim.sales_order_lines)

monthly_revenue = analytics_tables["monthly_sales_summary"]["booked_revenue"].sum()
raw_revenue = orders_df["order_total"].sum()

round(monthly_revenue - raw_revenue, 2)

In [ ]:
model_units = analytics_tables["model_performance_summary"]["requested_units"].sum()
raw_units = lines_df["requested_qty"].sum()

model_units - raw_units

In [ ]:
sim.export_tables()

In [ ]:
from simulation.database_builder import build_sqlite_database

database_result = build_sqlite_database()

database_result["database_path"]

In [ ]:
database_result["load_results"]

In [ ]:
database_result["table_counts"]

In [ ]:
import sqlite3
import pandas as pd

db_path = database_result["database_path"]

conn = sqlite3.connect(db_path)

In [ ]:
query = """
SELECT
    model_name,
    category,
    SUM(requested_qty) AS requested_units,
    ROUND(SUM(extended_price), 2) AS booked_revenue,
    ROUND(SUM(fulfilled_revenue), 2) AS fulfilled_revenue,
    SUM(backordered_qty) AS backordered_units,
    ROUND(
        CAST(SUM(fulfilled_qty) AS FLOAT) / NULLIF(SUM(requested_qty), 0),
        3
    ) AS service_level
FROM sales_order_lines
GROUP BY
    model_name,
    category
ORDER BY
    booked_revenue DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
query = """
SELECT
    c.region,
    c.state,
    so.sales_channel,
    COUNT(DISTINCT so.sales_order_id) AS sales_orders,
    SUM(sol.requested_qty) AS requested_units,
    ROUND(SUM(sol.extended_price), 2) AS booked_revenue,
    ROUND(SUM(sol.fulfilled_revenue), 2) AS fulfilled_revenue
FROM sales_orders so
JOIN customers c
    ON so.customer_id = c.customer_id
JOIN sales_order_lines sol
    ON so.sales_order_id = sol.sales_order_id
GROUP BY
    c.region,
    c.state,
    so.sales_channel
ORDER BY
    booked_revenue DESC;
"""

pd.read_sql_query(query, conn).head(20)

In [ ]:
conn.close()

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
products_path = PROJECT_ROOT / "data" / "products.csv"

products_df = pd.read_csv(products_path)

def assign_finished_goods_supplier(row):
    if row["frame_material"] == "Carbon":
        return "S001"  # Merida Industry

    if row["frame_material"] == "Aluminum":
        return "S002"  # Ideal Bike Corp

    return "S001"

products_df["supplier_id"] = products_df.apply(
    assign_finished_goods_supplier,
    axis=1
)

products_df.to_csv(products_path, index=False)

products_df["supplier_id"].value_counts()